In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv("powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [3]:
x=df.drop("PE", axis=1)
y=df["PE"]

In [4]:
#split the data
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,y,test_size=0.2,random_state=42)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [6]:
import torch
import torch.nn as nn

x_train_tensor=torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

x_test_tensor=torch.tensor(x_test_scaled, dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)


In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Buildin model

In [26]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(x_train.shape[1] ,6),
            nn.ReLU(),
            nn.Linear(6, 6),
            nn.ReLU(),
            nn.Linear(6,1),

        )

    def forward(self, x):
        return self.model(x)
        

In [28]:
import torch.optim as optim
model = ANN()
#loss, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

# training model

In [30]:
train_loss = []
val_loss=[]
epochs = 50

best_val_loss = float("inf")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        #xb= features ob batch 1
        #yb = labels of 1 batch
        optimizer.zero_grad()
        outputs=model(xb)# froward prop predicts output
        loss=crietrion(outputs, yb)# compute loss
        loss.backward()# backward prop
        optimizer.step()
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)
# validation
    model.eval()
    running_val_loss=0.0
    with torch.no_grad():
        for xb, yb in train_loader:
            #xb= features ob batch 1
            #yb = labels of 1 batch
            
            outputs=model(xb)# froward prop predicts output
            loss=crietrion(outputs, yb)# compute loss
            running_val_loss += loss
    epoch_val_loss = running_val_loss / len(train_loader)
    val_loss.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss= {epoch_train_loss} & val loss = {epoch_val_loss}")
    
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

    
    


epoch 1/50 ==> train loss= 205514.43352864584 & val loss = 203273.71875
epoch 2/50 ==> train loss= 194590.12057291667 & val loss = 181501.265625
epoch 3/50 ==> train loss= 161471.09680989583 & val loss = 139937.484375
epoch 4/50 ==> train loss= 122108.59847005208 & val loss = 108244.0
epoch 5/50 ==> train loss= 101479.09278971354 & val loss = 96576.359375
epoch 6/50 ==> train loss= 92958.0803548177 & val loss = 88868.8515625
epoch 7/50 ==> train loss= 83443.02014973959 & val loss = 76741.609375
epoch 8/50 ==> train loss= 66012.40844726562 & val loss = 51563.2265625
epoch 9/50 ==> train loss= 31825.452689615886 & val loss = 13857.67578125
epoch 10/50 ==> train loss= 5435.130599975586 & val loss = 1091.9046630859375
epoch 11/50 ==> train loss= 437.03391609191897 & val loss = 178.34559631347656
epoch 12/50 ==> train loss= 129.71318562825522 & val loss = 101.4295883178711
epoch 13/50 ==> train loss= 88.50303570429485 & val loss = 78.66365814208984
epoch 14/50 ==> train loss= 72.05952201684

In [37]:
# loading best model
model.load_state_dict(torch.load("best_model.pt"))


C:\Users\Nomaa\AppData\Local\Temp\ipykernel_2260\2772176056.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pt"))


<All keys matched successfully>

In [45]:
#evaluation

model.eval()
with torch.no_grad():
    train_preds=model(x_train_tensor)
    test_preds=model(x_test_tensor)

    train_mse_loss = crietrion(train_preds,y_train_tensor)
    test_mse_loss = crietrion(test_preds,y_test_tensor)

print("Training mse:", train_mse_loss)
print("Testing mse:", test_mse_loss)

Training mse: tensor(21.2231)
Testing mse: tensor(19.7095)


In [49]:
from sklearn.metrics import r2_score

print("r2 score=:", r2_score(y_test, test_preds))

r2 score=: 0.93112046406049
